# Cambodian legal OCR → article dataset

This notebook converts final OCR `.txt` files into one nested, search-ready JSON document per law. It preserves legal wording and useful document metadata, groups articles under chapters and sections, and preserves meaningful line breaks while allowing only one newline in a row.

**Important:** Article `text` is cleaned only for formatting noise. No translation, summarization, tokenization, embeddings, or silent OCR correction is performed.

## 1. Setup and paths

The notebook writes one JSON file per document under `data/processed/json`.

In [1]:
import json
import re
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'react-client').is_dir() and (path / 'data-pipeline').is_dir()
)
DATA_DIR = PROJECT_ROOT / 'data-pipeline' / 'data'
FINAL_TEXT_DIR = DATA_DIR / 'ocr' / 'text'
ORIGINAL_SOURCE_CSV = DATA_DIR / 'checkpoints' / 'documents.csv'
PIPELINE_TRACKING_CSV = DATA_DIR / 'checkpoints' / 'documents_pipeline.csv'
OUTPUT_DIR = DATA_DIR / 'processed' / 'json'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Project:', PROJECT_ROOT)
print('Final OCR files:', len(list(FINAL_TEXT_DIR.glob('*.txt'))))


Project: /Users/nak/CADT/projects/legal-rag
Final OCR files: 95


## 2. Load document metadata

The metadata file is read as strings so document IDs, URLs, Khmer titles, and empty values are preserved. Each final OCR filename (`{document_id}.txt`) is joined to this original source row by `document_id`; no metadata is guessed.

In [2]:
documents = pd.read_csv(
    ORIGINAL_SOURCE_CSV,
    dtype='string',
    keep_default_na=False,
    encoding='utf-8-sig',
)
documents['document_id'] = documents['document_id'].str.strip()
documents = documents.drop_duplicates('document_id', keep='last').set_index('document_id')
final_document_ids = {p.stem for p in FINAL_TEXT_DIR.glob('*.txt')}
missing_source_rows = final_document_ids - set(documents.index)
assert not missing_source_rows, (
    'Final OCR files missing original source rows: '
    f'{sorted(missing_source_rows)}'
)

def metadata_for(document_id):
    row = documents.loc[document_id].to_dict() if document_id in documents.index else {}
    metadata = {
        str(key): (None if pd.isna(value) else str(value).strip())
        for key, value in row.items()
    }
    for field in ('title', 'document_type', 'publication_date', 'publication_year', 'language', 'category'):
        if metadata.get(field):
            metadata[field] = ' '.join(unicode_clean(metadata[field]).split())
    metadata['document_id'] = str(document_id)
    return metadata

display(documents.reset_index().head(3))


,document_id,title,detail_url,pdf_url,local_pdf_path,status,retry_count,error,document_type,publication_date,publication_year,language,category,category_id,updated_at
0,1,ច្បាប់ស្តីពីចរាចរផ្លូវគោក​ (វិសោធនកម្ម),https://library.ncdd.gov.kh/detail/1,https://library.ncdd.gov.kh/detail/1/download,/opt/legal-rag/data/pdfs/1.pdf,downloaded,3,,Law,,2024,Khmer,NCDD Laws,7,2026-10-28T16:13:15.738615+00:00
1,1004,ច្បាប់ស្តីពីប្រកាសនីយបត្រតក្កកម្មវិញ្ញាបនបត្រម...,https://library.ncdd.gov.kh/detail/1004,https://lib.ncdd.gov.kh/storage/app/public/lib...,/opt/legal-rag/data/pdfs/1004.pdf,downloaded,6,,Law,,2024,Khmer,NCDD Laws,7,2026-10-28T16:13:33.870656+00:00
2,1006,ច្បាប់ស្តីពីសិទ្ធិអ្នកនិពន្ធ និងសិទ្ធិប្រហាក់ប...,https://library.ncdd.gov.kh/detail/1006,https://lib.ncdd.gov.kh/storage/app/public/lib...,/opt/legal-rag/data/pdfs/1006.pdf,downloaded,6,,Law,,2024,Khmer,NCDD Laws,7,2026-10-28T16:13:49.678443+00:00


## 3. Clean OCR formatting without changing legal meaning

OCR commonly contains zero-width characters, non-breaking spaces, repeated whitespace, and decorative formatting noise. The cleaning functions remove search-irrelevant punctuation such as `៖`, `។`, and `៕`, while preserving meaningful numbers, signs, and list boundaries.

In [3]:
KHMER_DIGITS = str.maketrans('០១២៣៤៥៦៧៨៩', '0123456789')
ZERO_WIDTH = '\u200b\u200c\u200d\ufeff'
DECORATIVE_ONLY_RE = re.compile(r'^\s*(?:[•+\-–—·]+|\.{2,})\s*$')
DECORATIVE_DOTS_RE = re.compile(r'\.{3,}')
# Keep commas: OCR uses them as decimal separators, for example ០,៤០.
SEARCH_NOISE_RE = re.compile(r'[៖:;!?។៕！？]')
PAGE_NUMBER_RE = re.compile(r'^\s*(?:[0-9០-៩]{3,}|[0-9០-៩]+\s*/\s*[0-9០-៩]+)\s*$')
PAGE_CONTEXT_RE = re.compile(r'[។៕]\s*$')


def unicode_clean(value):
    value = unicodedata.normalize('NFKC', str(value or ''))
    value = value.translate(str.maketrans('', '', ZERO_WIDTH))
    return value.replace('\u00a0', ' ')


def clean_line(value):
    line = unicode_clean(value).strip()
    if not line or DECORATIVE_ONLY_RE.fullmatch(line):
        return ''
    line = DECORATIVE_DOTS_RE.sub(' ', line)
    line = SEARCH_NOISE_RE.sub(' ', line)
    line = re.sub(r'[ \t]+', ' ', line)
    return line.strip()


def remove_page_number_lines(lines):
    result = []
    for index, item in enumerate(lines):
        raw_line = unicode_clean(item).strip()
        if not PAGE_NUMBER_RE.fullmatch(raw_line):
            result.append(item)
            continue

        previous = next((unicode_clean(lines[pos]).strip() for pos in range(index - 1, -1, -1) if unicode_clean(lines[pos]).strip()), '')
        following = next((unicode_clean(lines[pos]).strip() for pos in range(index + 1, len(lines)) if unicode_clean(lines[pos]).strip()), '')
        blank_before = index > 0 and not unicode_clean(lines[index - 1]).strip()
        blank_after = index + 1 < len(lines) and not unicode_clean(lines[index + 1]).strip()
        is_page_number = bool(
            previous and (PAGE_CONTEXT_RE.search(previous) or blank_before or blank_after)
        )
        if not is_page_number:
            result.append(item)
    return result


def clean_multiline_text(lines):
    # Preserve meaningful OCR line breaks, but keep only one newline in a row.
    cleaned_lines = [clean_line(item) for item in remove_page_number_lines(lines)]
    text = '\n'.join(line for line in cleaned_lines if line).strip()
    return re.sub(r'\n{2,}', '\n', text)


def clean_title(value):
    return ' '.join(clean_line(value).split())


def number_value(value):
    digits = unicode_clean(value).translate(KHMER_DIGITS)
    return int(digits) if digits.isdigit() else None


## 4. Detect chapters, sections, and articles

The article boundary is the key unit for legal search. The regex allows spaces and common OCR punctuation, for example `មាត្រា១`, `មាត្រា ១៨._ ថ្មី`, and `មាត្រា 5.-`. Text before the first article is retained as `preamble` rather than silently discarded.

In [4]:
NUM = r'[0-9០-៩]+'
ARTICLE_NUMBER_TEXT = r'(?:' + NUM + r'|មួយ|ពីរ|បី|បួន|ប្រាំ(?:មួយ|ពីរ|បី|បួន|ប្រាំ)?)'
CHAPTER_RE = re.compile(r'^\s*(?:ផ្នែក\s*[:：]\s*)?ជំពូក\s*ទី\s*(' + NUM + r')\s*(?:[._៖:-]+)?\s*(.*)$')
SECTION_RE = re.compile(r'^\s*ផ្នែក\s*ទី\s*(' + NUM + r')\s*(?:[._៖:-]+)?\s*(.*)$')
ARTICLE_TOKEN_RE = re.compile(r'\[START_ARTICLE\s*:\s*(' + NUM + r')\s*\]')
ARTICLE_MARKER_RE = re.compile(r'^\s*\[START_ARTICLE\s*:\s*(' + NUM + r')\s*\]\s*$')
ARTICLE_HEADING_RE = re.compile(r'^\s*["“”]?\s*មាត្រា\s*(' + NUM + r')(?:\s|[._៖:-]|$)', re.IGNORECASE)
DOCUMENT_NUMBER_RE = re.compile(r'(?m)^\s*([^\s/]+(?:\s*/\s*[^\s/]+){3})\s*$')


def heading_match(pattern, line):
    match = pattern.match(unicode_clean(line))
    if not match:
        return None
    return number_value(match.group(1)), clean_title(match.group(2))


def remove_article_heading(lines):
    cleaned = [clean_line(line) for line in remove_page_number_lines(lines)]
    cleaned = [line for line in cleaned if line]
    if cleaned:
        cleaned[0] = re.sub(
            r'^["“”]?\s*មាត្រា\s*' + ARTICLE_NUMBER_TEXT + r'\s*(?:[._៖:-]+\s*)?',
            '',
            cleaned[0],
            count=1,
        ).strip()
        if not cleaned[0]:
            cleaned.pop(0)
    return cleaned


APPENDIX_START_RE = re.compile(r'^\s*(?:[0-9០-៩]+\s*/\s*[0-9០-៩]+|ការគ្រប់គ្រងឆ្នាំ|តារាង\s*["“]|ឧបសម្ព័ន្ធ)')
CLOSING_START_RE = re.compile(r'^\s*(?:ធ្វើនៅ.*ថ្ងៃទី|រាជធានីភ្នំពេញ.*ថ្ងៃទី|ព្រះហស្តលេខា(?:\s|$))')
OFFICIAL_COPY_DATE_RE = re.compile(r'^\s*រាជធានីភ្នំពេញ.*ថ្ងៃទី')


def next_nonempty(lines, start):
    for index in range(start, len(lines)):
        value = clean_title(lines[index])
        if value:
            return value
    return None


def extract_closing_information(lines):
    cleaned = [clean_line(line) for line in lines]
    signing_index = next(
        (index for index, line in enumerate(cleaned) if CLOSING_START_RE.match(line)),
        None,
    )
    if signing_index is None:
        return cleaned, None

    closing_lines = []
    for line in cleaned[signing_index:]:
        if APPENDIX_START_RE.match(line):
            break
        if line:
            closing_lines.append(line)

    signing = {'signed_at': clean_title(closing_lines[0])}
    official_copy_date = next(
        (clean_title(line) for line in closing_lines if OFFICIAL_COPY_DATE_RE.match(line)),
        None,
    )
    if official_copy_date:
        signing['official_copy_date'] = official_copy_date

    king_marker = next(
        (index for index, line in enumerate(closing_lines) if 'ព្រះហស្តលេខា' in line),
        None,
    )
    king_name = next_nonempty(closing_lines, king_marker + 1) if king_marker is not None else None
    if king_name and 'នរោត្តម' in king_name:
        signing['king'] = king_name

    prime_marker = next(
        (index for index, line in enumerate(closing_lines) if line == 'នាយករដ្ឋមន្ត្រី'),
        None,
    )
    if prime_marker is not None:
        prime_name = next(
            (clean_title(closing_lines[index + 1]) for index in range(prime_marker + 1, len(closing_lines))
             if closing_lines[index] == 'ហត្ថលេខា' and index + 1 < len(closing_lines)),
            None,
        )
        if prime_name:
            signing['prime_minister'] = prime_name

    minister_marker = next(
        (index for index, line in enumerate(closing_lines) if 'រដ្ឋមន្ត្រីក្រសួង' in line),
        None,
    )
    if minister_marker is not None:
        minister_name = next_nonempty(closing_lines, minister_marker + 1)
        if minister_name == 'ហត្ថលេខា':
            minister_name = next_nonempty(closing_lines, minister_marker + 2)
        if minister_name:
            signing['minister'] = minister_name

    secretary_marker = next(
        (index for index, line in enumerate(closing_lines) if line == 'អគ្គលេខាធិការរាជរដ្ឋាភិបាល'),
        None,
    )
    secretary_name = next_nonempty(closing_lines, secretary_marker + 1) if secretary_marker is not None else None
    if secretary_name:
        signing['government_secretary'] = secretary_name

    return cleaned[:signing_index], signing


def source_text(document_id):
    path = FINAL_TEXT_DIR / f'{document_id}.txt'
    if not path.is_file():
        raise FileNotFoundError(f'Final OCR text is missing: {path}')
    return path.read_text(encoding='utf-8-sig'), str(path)


def extract_document_number(text):
    normalized = unicode_clean(text)
    matches = list(DOCUMENT_NUMBER_RE.finditer(normalized))
    if not matches:
        return None

    first_article = normalized.find('[START_ARTICLE:')
    before_articles = [match for match in matches if first_article < 0 or match.start() < first_article]
    # The number is normally at the beginning. If OCR places it after the
    # article content, use the final standalone number instead.
    match = before_articles[0] if before_articles else matches[-1]
    return re.sub(r'\s*/\s*', '/', match.group(1)).strip()


In [5]:
def parse_document(text):
    normalized_text = ARTICLE_TOKEN_RE.sub(
        lambda match: f'\n[START_ARTICLE:{match.group(1)}]\n',
        unicode_clean(text),
    )
    if not ARTICLE_TOKEN_RE.search(normalized_text):
        normalized_text = re.sub(
            r'(?m)^\s*(?=["“”]?\s*មាត្រា\s*(' + NUM + r')(?:\s|[._៖:-]|$))',
            lambda match: f'\n[START_ARTICLE:{match.group(1)}]\n',
            normalized_text,
        )
    preamble_lines = []
    chapters = []
    ungrouped_articles = []
    current_chapter = None
    current_section = None
    current_article = None
    pending_heading = None

    def ensure_chapter():
        nonlocal current_chapter
        if current_chapter is None:
            current_chapter = {
                'chapter_number': None,
                'chapter_title': '',
                'sections': [],
                'articles': [],
            }
            chapters.append(current_chapter)
        return current_chapter

    def save_article():
        nonlocal current_article
        if current_article is None:
            return
        article_lines = remove_article_heading(current_article['lines'])
        article_lines, _signing = extract_closing_information(article_lines)
        body = clean_multiline_text(article_lines)
        if body:
            article = {
                'article_number': current_article['article_number'],
                'text': body,
            }
            if current_section is not None:
                current_section['articles'].append(article)
            elif current_chapter is not None:
                current_chapter['articles'].append(article)
            else:
                ungrouped_articles.append(article)
        current_article = None

    for raw_line in normalized_text.splitlines():
        line = raw_line.strip()
        chapter = heading_match(CHAPTER_RE, line)
        section = heading_match(SECTION_RE, line)
        marker = ARTICLE_MARKER_RE.match(line)

        # A chapter or section title may be on the same line as its number
        # or on the immediately following non-empty line.
        if pending_heading is not None and line:
            if marker or chapter or section:
                pending_heading = None
            else:
                pending_heading['target']['%s_title' % pending_heading['kind']] = clean_title(line)
                pending_heading = None
                continue

        if marker:
            save_article()
            current_article = {
                'article_number': number_value(marker.group(1)),
                'lines': [],
            }
        elif chapter:
            save_article()
            chapter_number, chapter_title = chapter
            current_chapter = {
                'chapter_number': chapter_number,
                'chapter_title': chapter_title,
                'sections': [],
                'articles': [],
            }
            chapters.append(current_chapter)
            if not chapter_title:
                pending_heading = {'kind': 'chapter', 'target': current_chapter}
            current_section = None
        elif section:
            save_article()
            chapter = ensure_chapter()
            section_number, section_title = section
            current_section = {
                'section_number': section_number,
                'section_title': section_title,
                'articles': [],
            }
            chapter['sections'].append(current_section)
            if not section_title:
                pending_heading = {'kind': 'section', 'target': current_section}
        elif current_article is not None:
            current_article['lines'].append(raw_line)
        elif line:
            preamble_lines.append(raw_line)

    save_article()
    return clean_multiline_text(preamble_lines), chapters, ungrouped_articles


## 5. Build nested JSON documents

Each output document preserves metadata, preamble, chapters, sections, and articles. Article content is stored in a cleaned `text` field with meaningful line breaks preserved. Document URLs remain available for citation and source access; internal pipeline paths and processing state are excluded.

In [6]:
def build_document(document_id):
    text, _text_path = source_text(document_id)
    metadata = metadata_for(document_id)
    preamble, chapters, ungrouped_articles = parse_document(text)

    metadata['document_number'] = extract_document_number(text)
    public_fields = [
        'document_id',
        'title',
        'document_number',
        'document_type',
        'publication_date',
        'publication_year',
        'language',
        'category',
        'detail_url',
        'pdf_url',
    ]
    public_metadata = {
        field: metadata.get(field)
        for field in public_fields
        if metadata.get(field) not in (None, '')
    }

    document = {
        **public_metadata,
        'preamble': preamble,
        'chapters': chapters,
    }
    if ungrouped_articles:
        document['articles'] = ungrouped_articles
    return document

document_ids = sorted(p.stem for p in FINAL_TEXT_DIR.glob('*.txt'))
all_documents_json = [build_document(document_id) for document_id in document_ids]

def article_count(document):
    count = len(document.get('articles', []))
    for chapter in document.get('chapters', []):
        count += len(chapter.get('articles', []))
        for section in chapter.get('sections', []):
            count += len(section.get('articles', []))
    return count

documents_json = [document for document in all_documents_json if article_count(document) > 0]
processed_by_id = {document['document_id']: document for document in documents_json}
pipeline_time = datetime.now(timezone.utc).isoformat()
tracking_rows = []
for document_id, row in documents.reset_index().set_index('document_id').iterrows():
    document_id = str(document_id)
    article_total = article_count(processed_by_id[document_id]) if document_id in processed_by_id else 0
    has_ocr = document_id in final_document_ids
    if not has_ocr:
        pipeline_status = 'not_processed'
        pipeline_note = 'Final OCR text is missing'
    elif article_total == 0:
        pipeline_status = 'skipped'
        pipeline_note = 'No reliable article structure detected'
    else:
        pipeline_status = 'processed'
        pipeline_note = 'Article JSON generated'
    tracking_row = row.to_dict()
    tracking_row.update({
        'document_id': document_id,
        'pipeline_status': pipeline_status,
        'pipeline_note': pipeline_note,
        'article_count': article_total,
        'json_path': f'data/processed/json/{document_id}.json' if article_total else '',
        'pipeline_updated_at': pipeline_time,
    })
    tracking_rows.append(tracking_row)
tracking = pd.DataFrame(tracking_rows)
source_columns = documents.reset_index().columns.tolist()
pipeline_columns = ['pipeline_status', 'pipeline_note', 'article_count', 'json_path', 'pipeline_updated_at']
tracking = tracking.reindex(columns=source_columns + pipeline_columns)
tracking.to_csv(PIPELINE_TRACKING_CSV, index=False, encoding='utf-8-sig')
print(f'{len(documents_json):,} documents prepared for JSON output')
print(f'{sum(row["pipeline_status"] == "skipped" for row in tracking_rows):,} documents skipped')
print('Tracking file:', PIPELINE_TRACKING_CSV)
print(json.dumps(documents_json[0], ensure_ascii=False, indent=2)[:2500])


90 documents prepared for JSON output
5 documents skipped
Tracking file: /Users/nak/CADT/projects/legal-rag/data-pipeline/data/checkpoints/documents_pipeline.csv
{
  "document_id": "1",
  "title": "ច្បាប់ស្តីពីចរាចរផ្លូវគោក (វិសោធនកម្ម)",
  "document_number": "នស/រកម/០១១៧/០០២",
  "document_type": "Law",
  "publication_year": "2024",
  "language": "Khmer",
  "category": "NCDD Laws",
  "detail_url": "https://library.ncdd.gov.kh/detail/1",
  "pdf_url": "https://library.ncdd.gov.kh/detail/1/download",
  "preamble": "ព្រះរាជក្រម\nនស/រកម/០១១៧/០០២\nយើង\nព្រះករុណាព្រះបាទសម្តេចព្រះបរមនាថ នរោត្តម សីហមុនី\nសមានភូមិជាតិសាសនា រក្ខតខត្តិយា ខេមរារដ្ឋរាស្ត្រ ពុទ្ធិន្ទ្រាធរាមហាក្សត្រ\nខេមរាជនា សមូហោភាស កម្ពុជឯករាជរដ្ឋបូរណសន្តិ សុភមង្គលា សិរីវិបុលា\nខេមរាស្រីពិរាស្ត្រ ព្រះចៅក្រុងកម្ពុជាធិបតី\n- បានទ្រង់យល់រដ្ឋធម្មនុញ្ញនៃព្រះរាជាណាចក្រកម្ពុជា\n- បានទ្រង់យល់ព្រះរាជក្រឹត្យលេខ នស/រកត/០៩១៣/៩០៣ ចុះថ្ងៃទី២៤ ខែកញ្ញា ឆ្នាំ២០១៣ ស្តីពីការតែងតាំងរាជរដ្ឋាភិបាលនៃព្រះរាជាណាចក្រកម្ពុជា\n- បានទ្រង់យល់ព្រះរាជក្រមលេខ ០២/ន

## 6. Quality checks before export

These checks catch OCR/parser failures early. Duplicate article numbers are allowed because they can occur in different documents; duplicates within one document deserve review.

In [7]:
assert documents_json, 'No documents were parsed'
for document in documents_json:
    assert document['document_id'], 'Document ID is missing'
    assert isinstance(document['chapters'], list), 'Chapters must be a list'
    for chapter in document['chapters']:
        assert isinstance(chapter.get('articles', []), list)
        for article in chapter.get('articles', []):
            assert article['text'], 'Article text must not be empty'
            assert '\n\n' not in article['text'], 'Article text contains repeated newlines'
        for section in chapter.get('sections', []):
            for article in section.get('articles', []):
                assert article['text'], 'Section article text must not be empty'
                assert '\n\n' not in article['text'], 'Article text contains repeated newlines'
print('JSON structure validation passed')


JSON structure validation passed


## 7. Export JSON files

One valid UTF-8 JSON file is written per document under `data/processed/json`. Preamble text is stored at the document level, while articles remain grouped under their chapters and sections. No embeddings, tokenization, or search-specific fields are created here.

In [8]:
written_paths = []
for document in documents_json:
    output_path = OUTPUT_DIR / f"{document['document_id']}.json"
    output_path.write_text(
        json.dumps(document, ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )
    written_paths.append(output_path)
print(f'Wrote {len(written_paths):,} JSON documents to {OUTPUT_DIR}')


Wrote 90 JSON documents to /Users/nak/CADT/projects/legal-rag/data-pipeline/data/processed/json


## 8. JSON smoke test

The final check confirms that every JSON document has an identifier, valid hierarchy, non-empty article text, and no repeated consecutive newline characters. Later stages can use these JSON documents to build lexical indexes, embeddings, and vector search structures.

In [9]:
sample_path = written_paths[0]
print(sample_path)
print(sample_path.read_text(encoding='utf-8')[:2500])


/Users/nak/CADT/projects/legal-rag/data-pipeline/data/processed/json/1.json
{
  "document_id": "1",
  "title": "ច្បាប់ស្តីពីចរាចរផ្លូវគោក (វិសោធនកម្ម)",
  "document_number": "នស/រកម/០១១៧/០០២",
  "document_type": "Law",
  "publication_year": "2024",
  "language": "Khmer",
  "category": "NCDD Laws",
  "detail_url": "https://library.ncdd.gov.kh/detail/1",
  "pdf_url": "https://library.ncdd.gov.kh/detail/1/download",
  "preamble": "ព្រះរាជក្រម\nនស/រកម/០១១៧/០០២\nយើង\nព្រះករុណាព្រះបាទសម្តេចព្រះបរមនាថ នរោត្តម សីហមុនី\nសមានភូមិជាតិសាសនា រក្ខតខត្តិយា ខេមរារដ្ឋរាស្ត្រ ពុទ្ធិន្ទ្រាធរាមហាក្សត្រ\nខេមរាជនា សមូហោភាស កម្ពុជឯករាជរដ្ឋបូរណសន្តិ សុភមង្គលា សិរីវិបុលា\nខេមរាស្រីពិរាស្ត្រ ព្រះចៅក្រុងកម្ពុជាធិបតី\n- បានទ្រង់យល់រដ្ឋធម្មនុញ្ញនៃព្រះរាជាណាចក្រកម្ពុជា\n- បានទ្រង់យល់ព្រះរាជក្រឹត្យលេខ នស/រកត/០៩១៣/៩០៣ ចុះថ្ងៃទី២៤ ខែកញ្ញា ឆ្នាំ២០១៣ ស្តីពីការតែងតាំងរាជរដ្ឋាភិបាលនៃព្រះរាជាណាចក្រកម្ពុជា\n- បានទ្រង់យល់ព្រះរាជក្រមលេខ ០២/នស/៩៤ ចុះថ្ងៃទី២០ ខែកក្កដា ឆ្នាំ១៩៩៤ ដែលប្រកាសឱ្យប្រើច្បាប់ស្តីពីការរៀបចំ និងការប្រព្រឹ